In [6]:
# ==========================================
# CELL 1: THE DEPENDENCY RESET
# ==========================================
print("♻️  Cleaning up the Python Environment...")

# 1. Uninstall the conflicting stack
!pip uninstall -y numpy scipy scikit-learn pandas chromadb sentence-transformers

# 2. Re-install with strict compatibility constraints
# We force numpy < 2.0, and let pip find the matching scipy/sklearn versions for it.
print("⬇️  Installing compatible versions...")
!pip install "numpy<2.0.0" "scipy<1.13.0" "scikit-learn<1.5.0" sentence-transformers chromadb langchain-community langchain-huggingface

print("✅ Dependencies reset.")
print("⚠️ PLEASE RESTART RUNTIME NOW (Runtime -> Restart Session).")

♻️  Cleaning up the Python Environment...
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.12.0
Uninstalling scipy-1.12.0:
  Successfully uninstalled scipy-1.12.0
Found existing installation: scikit-learn 1.4.2
Uninstalling scikit-learn-1.4.2:
  Successfully uninstalled scikit-learn-1.4.2
Found existing installation: chromadb 1.3.5
Uninstalling chromadb-1.3.5:
  Successfully uninstalled chromadb-1.3.5
Found existing installation: sentence-transformers 5.1.2
Uninstalling sentence-transformers-5.1.2:
  Successfully uninstalled sentence-transformers-5.1.2
⬇️  Installing compatible versions...
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached scikit_learn-1.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metad

✅ Dependencies reset.
⚠️ PLEASE RESTART RUNTIME NOW (Runtime -> Restart Session).


In [1]:
# ==========================================
# CELL 2: IMPORTS & DRIVE
# ==========================================
import os
import pickle
import shutil
import gc
import torch
from google.colab import drive

# Mount Drive
if not os.path.exists('/content/drive'):
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted.")

# Check GPU
if torch.cuda.is_available():
    print(f"🎮 GPU Detected: {torch.cuda.get_device_name(0)}")
    DEVICE = 'cuda'
else:
    print("⚠️ No GPU detected. This will be slow.")
    DEVICE = 'cpu'

# Path Configuration
BASE_PATH = "/content/drive/MyDrive/GenAiProject_Dataset"
PICKLE_FILE = os.path.join(BASE_PATH, "processed_chunks_500.pkl")

✅ Drive already mounted.
🎮 GPU Detected: Tesla T4


In [2]:
# ==========================================
# CELL 3: DATA CLEANING (THE FIX)
# ==========================================
def robust_load_and_clean(pickle_path):
    print(f"📂 Loading chunks from: {pickle_path}")

    if not os.path.exists(pickle_path):
        print(f"❌ Error: File not found at {pickle_path}")
        return [], []

    with open(pickle_path, "rb") as f:
        chunks = pickle.load(f)

    valid_texts = []
    valid_metadatas = []
    dropped_count = 0

    print(f"🧹 Scrubbing {len(chunks)} chunks for bad data...")

    for i, doc in enumerate(chunks):
        # 1. Extract Content Safely
        content = getattr(doc, 'page_content', None)

        # 2. Force Conversion to String
        if content is None:
            text = ""
        else:
            text = str(content)

        # 3. Clean Whitespace & Null Bytes
        text = text.strip().replace('\x00', '')

        # 4. Final Validation: Must be non-empty
        if len(text) > 0:
            valid_texts.append(text)
            # Safe Metadata Extraction
            meta = getattr(doc, 'metadata', {})
            # Ensure all metadata values are strings (ChromaDB requirement)
            clean_meta = {k: str(v) for k, v in meta.items()}
            valid_metadatas.append(clean_meta)
        else:
            dropped_count += 1

    print(f"   ✓ Kept {len(valid_texts)} valid text chunks.")
    print(f"   🗑️ Dropped {dropped_count} empty/bad chunks.")

    return valid_texts, valid_metadatas

# RUN THE CLEANER
texts, metadatas = robust_load_and_clean(PICKLE_FILE)

📂 Loading chunks from: /content/drive/MyDrive/GenAiProject_Dataset/processed_chunks_500.pkl
🧹 Scrubbing 2108 chunks for bad data...
   ✓ Kept 2108 valid text chunks.
   🗑️ Dropped 0 empty/bad chunks.


In [4]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 125.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.12.0 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


In [5]:
# ==========================================
# CELL 4: ROBUST BASELINE (FIXED)
# ==========================================
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

def safe_generate_embeddings(model, texts):
    try:
        return model.encode(texts, convert_to_numpy=True).tolist()
    except Exception as e:
        print(f"   ⚠️ Batch crash! Switching to safe mode...")
        valid_embeddings = []
        valid_indices = []
        for i, text in enumerate(texts):
            try:
                emb = model.encode(text, convert_to_numpy=True).tolist()
                valid_embeddings.append(emb)
                valid_indices.append(i)
            except:
                pass # Skip bad data silently
        return valid_embeddings, valid_indices

def build_baseline_robust(texts, metadatas):
    MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
    SAVE_FOLDER = "vector_db_baseline_500"
    SAVE_PATH = os.path.join(BASE_PATH, SAVE_FOLDER)

    print(f"\n🚀 BUILDING BASELINE (ROBUST MODE): {MODEL_NAME}")

    model = SentenceTransformer(MODEL_NAME, device=DEVICE)

    # --- FIX: SAFE DATABASE RESET ---
    client = chromadb.PersistentClient(path=SAVE_PATH)
    try:
        client.delete_collection("documents") # Force delete inside DB
        print("   ✓ Cleared old zombie collection.")
    except:
        pass # Collection didn't exist, which is fine

    collection = client.create_collection(name="documents", metadata={"hnsw:space": "cosine"})
    # --------------------------------

    BATCH_SIZE = 32
    total = len(texts)

    for i in range(0, total, BATCH_SIZE):
        end = min(i + BATCH_SIZE, total)
        batch_texts = texts[i:end]
        batch_metas = metadatas[i:end]
        batch_ids = [f"id_{j}" for j in range(i, end)]

        result = safe_generate_embeddings(model, batch_texts)

        if isinstance(result, tuple):
            batch_embeddings, valid_indices = result
            batch_texts = [batch_texts[k] for k in valid_indices]
            batch_metas = [batch_metas[k] for k in valid_indices]
            batch_ids = [batch_ids[k] for k in valid_indices]
        else:
            batch_embeddings = result

        if len(batch_embeddings) > 0:
            collection.add(
                ids=batch_ids,
                documents=batch_texts,
                embeddings=batch_embeddings,
                metadatas=batch_metas
            )

        if i % 100 == 0:
            print(f"   ✓ Processed {end}/{total}...")

    print(f"✅ SUCCESS! Baseline Database saved to: {SAVE_FOLDER}")
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Run it
if len(texts) > 0:
    build_baseline_robust(texts, metadatas)


🚀 BUILDING BASELINE (ROBUST MODE): sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✓ Processed 32/2108...
   ⚠️ Batch crash! Switching to safe mode...
   ⚠️ Batch crash! Switching to safe mode...
   ✓ Processed 832/2108...
   ⚠️ Batch crash! Switching to safe mode...
   ⚠️ Batch crash! Switching to safe mode...
   ✓ Processed 1632/2108...
✅ SUCCESS! Baseline Database saved to: vector_db_baseline_500


In [6]:
# ==========================================
# CELL 5: ROBUST ADVANCED (FIXED)
# ==========================================
def build_advanced_robust(texts, metadatas):
    MODEL_NAME = "BAAI/bge-base-en-v1.5"
    SAVE_FOLDER = "vector_db_advanced_500"
    SAVE_PATH = os.path.join(BASE_PATH, SAVE_FOLDER)

    print(f"\n🚀 BUILDING ADVANCED (ROBUST MODE): {MODEL_NAME}")

    model = SentenceTransformer(MODEL_NAME, device=DEVICE)

    # --- FIX: SAFE DATABASE RESET ---
    client = chromadb.PersistentClient(path=SAVE_PATH)
    try:
        client.delete_collection("documents")
        print("   ✓ Cleared old zombie collection.")
    except:
        pass

    collection = client.create_collection(name="documents", metadata={"hnsw:space": "cosine"})
    # --------------------------------

    BATCH_SIZE = 32
    total = len(texts)

    for i in range(0, total, BATCH_SIZE):
        end = min(i + BATCH_SIZE, total)
        batch_texts = texts[i:end]
        batch_metas = metadatas[i:end]
        batch_ids = [f"id_{j}" for j in range(i, end)]

        try:
            batch_embeddings = model.encode(
                batch_texts,
                normalize_embeddings=True,
                convert_to_numpy=True
            ).tolist()
        except:
            print(f"   ⚠️ Batch crash in BGE! Filter mode activated...")
            batch_embeddings = []
            valid_indices = []
            for k, text in enumerate(batch_texts):
                try:
                    emb = model.encode(text, normalize_embeddings=True, convert_to_numpy=True).tolist()
                    batch_embeddings.append(emb)
                    valid_indices.append(k)
                except:
                    pass

            batch_texts = [batch_texts[k] for k in valid_indices]
            batch_metas = [batch_metas[k] for k in valid_indices]
            batch_ids = [batch_ids[k] for k in valid_indices]

        if len(batch_embeddings) > 0:
            collection.add(
                ids=batch_ids,
                documents=batch_texts,
                embeddings=batch_embeddings,
                metadatas=batch_metas
            )

        if i % 100 == 0:
            print(f"   ✓ Processed {end}/{total}...")

    print(f"✅ SUCCESS! Advanced Database saved to: {SAVE_FOLDER}")
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Run it
if len(texts) > 0:
    build_advanced_robust(texts, metadatas)


🚀 BUILDING ADVANCED (ROBUST MODE): BAAI/bge-base-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✓ Processed 32/2108...
   ⚠️ Batch crash in BGE! Filter mode activated...
   ⚠️ Batch crash in BGE! Filter mode activated...
   ✓ Processed 832/2108...
   ⚠️ Batch crash in BGE! Filter mode activated...
   ⚠️ Batch crash in BGE! Filter mode activated...
   ✓ Processed 1632/2108...
✅ SUCCESS! Advanced Database saved to: vector_db_advanced_500


In [8]:
# ==========================================
# CELL 6: TEST YOUR TWO BRAINS (FIXED)
# ==========================================
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd
import os

# CONFIG (Re-define just in case)
BASE_PATH = "/content/drive/MyDrive/GenAiProject_Dataset"

# 1. Setup Retrieval Function
def get_retriever(folder_name, model_name):
    print(f"   ↳ Loading {model_name}...")

    # Re-initialize the embedding model
    model_kwargs = {'device': 'cuda'}
    encode_kwargs = {'normalize_embeddings': True} if "bge" in model_name else {}

    emb = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )

    # Load the DB
    db_path = os.path.join(BASE_PATH, folder_name)

    # --- THE FIX IS HERE ---
    # We must match the name we used in the builder ("documents")
    db = Chroma(
        persist_directory=db_path,
        embedding_function=emb,
        collection_name="documents"
    )

    # Return a retriever that finds top 3 results
    return db.as_retriever(search_kwargs={"k": 3})

print("⚙️ Loading Retrievers... (This takes 30 seconds)")
try:
    retriever_A = get_retriever("vector_db_baseline_500", "sentence-transformers/all-MiniLM-L6-v2")
    retriever_B = get_retriever("vector_db_advanced_500", "BAAI/bge-base-en-v1.5")
    print("✅ Models Loaded Successfully!")
except Exception as e:
    print(f"❌ Error loading models: {e}")

# 2. Define the Test
def compare_models(query):
    print(f"\n🔎 QUESTION: '{query}'")
    print("="*80)

    # Ask Model A
    print(f"🤖 BASELINE (MiniLM) Found:")
    try:
        results_A = retriever_A.invoke(query)
        if len(results_A) == 0:
            print("   [No results found - check data]")
        for i, doc in enumerate(results_A):
            source = os.path.basename(doc.metadata.get('source', 'Unknown'))
            print(f"   {i+1}. [{source}] \"{doc.page_content[:150]}...\"")
    except Exception as e:
        print(f"   ⚠️ Error: {e}")

    print("-" * 80)

    # Ask Model B
    print(f"🧠 ADVANCED (BGE) Found:")
    try:
        results_B = retriever_B.invoke(query)
        if len(results_B) == 0:
            print("   [No results found - check data]")
        for i, doc in enumerate(results_B):
            source = os.path.basename(doc.metadata.get('source', 'Unknown'))
            print(f"   {i+1}. [{source}] \"{doc.page_content[:150]}...\"")
    except Exception as e:
        print(f"   ⚠️ Error: {e}")
    print("="*80)

# 3. RUN THE TEST
# Change this question to something specific to your slides!
compare_models("What is the architecture of a Transformer?")

⚙️ Loading Retrievers... (This takes 30 seconds)
   ↳ Loading sentence-transformers/all-MiniLM-L6-v2...
   ↳ Loading BAAI/bge-base-en-v1.5...
✅ Models Loaded Successfully!

🔎 QUESTION: 'What is the architecture of a Transformer?'
🤖 BASELINE (MiniLM) Found:
   1. [Lecture # 9-1 Introduction to Transformers.pptx] "Stacked layers: Transformers typically consist of multiple layers stacked on top of each other. Each layer processes the output of the previous layer,..."
   2. [Lecture # 9-2 Vision Transformers.pptx] "Stacked layers: Transformers typically consist of multiple layers stacked on top of each other. Each layer processes the output of the previous layer,..."
   3. [Lecture # 9-1 Introduction to Transformers.pptx] "Training: Transformer models are trained using supervised learning, where they learn to minimize a loss function that quantifies the difference betwee..."
--------------------------------------------------------------------------------
🧠 ADVANCED (BGE) Found:
   1. [Lect